# HW2: Word2Vector
## Start

In [ ]:
%run CBOW.ipynb
# start word2vec train process from here
txt_path = "data/"
save_vec_path = "model/word2vec/"
save_model_path = "model/"
zh_stop_path = "data/hit_stopwords.txt"
train_path = "data/train/"

In [ ]:
# load text
zh_text = load_data(txt_path, version='zh')
# create words
zh_corpus = preprocess_text(zh_text, zh_stop_path=zh_stop_path, language='zh')
# create vocab
zh_vocab = build_vocab(zh_corpus)

In [ ]:
# create training data
zh_train_data = create_training_data(zh_corpus, zh_vocab, window_size=5)
torch.save(zh_train_data, train_path+"zh_train_data.pth")
# load training data
zh_train_data = torch.load(train_path+"zh_train_data.pth")
zh_idx2word = {v:k for k,v in zh_vocab.items()}
print(zh_train_data[0], '\n', [zh_idx2word.get(idx) for idx in zh_train_data[0][0]], '\n', zh_idx2word.get(zh_train_data[0][1]))

In [ ]:
# train zh model
model_zh = CBOW(vocab_size=len(zh_vocab), embedding_size=200).to(device)
# optimizer & loss function
optimizer, criterion = get_optim_and_loss(model_zh, "SGD", "cross")
print(model_zh, optimizer, criterion)

In [ ]:
epochs = 600
# set train mode
model_zh.train()
# start train
lossv_zh = []
print('Start zh training...')
train(model_zh, zh_train_data, optimizer, criterion, epochs, lossv_zh, batch_size=32)

In [ ]:
# plot 
plt.plot([epoch+1 for epoch in range(0, epochs, 100)], lossv_zh)
plt.title('CBOW Training Loss (zh)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

In [ ]:
# save
save_model(model_zh, save_model_path, version='zh')
save_word_vectors(model_zh, zh_vocab, save_vec_path, version='zh')

## 7. An Example Task to Use Trained Word Vectors
### Compute similarity among words

In [ ]:
# load word vectors
word_vectors_zh = load_word_vectors(save_vec_path, version="zh")

top_n = 3
zh_word = "人民"
# chinese
top_similarities_zh = find_similar_words(zh_word, word_vectors_zh, top_n)
print(f"Most similar top {top_n} words to '{zh_word}':")
for word, similarity in top_similarities_zh:
    print(f"{word}: {similarity:.3f}")

In [ ]:
# plot
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
principalComponents = pca.fit_transform(model_zh.embedding.weight.cpu().detach().numpy())

word2ReduceDimensionVec = {}
for word in zh_vocab.keys():
    word2ReduceDimensionVec[word] = principalComponents[zh_vocab[word], :]
    
plt.figure(figsize=(10, 10))
count = 0
for word, wordvec in word2ReduceDimensionVec.items():
    if count < 100:
        plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
        plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号，否则负号会显示成方块
        plt.scatter(wordvec[0], wordvec[1])
        plt.annotate(word, (wordvec[0], wordvec[1]))
        count += 1
plt.show()